# EDA: Панельные данные по группам грибов

5 групп: **болетовые** (белый, подосиновик, подберёзовик, маслёнок, моховик), **лисичковые** (июнь-сентябрь), **лисичковые трубчатые** (октябрь-ноябрь), **весенние** (сморчки, строчки — апрель-май), **опята** (июнь-октябрь).

Целевая: `mushroom_count` — нормализован по аудитории, дню недели, сглажен 3 дня.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option('display.max_colwidth', 200)

panel = pd.read_csv('data/panel.csv', parse_dates=['date'])
panel['year'] = panel['date'].dt.year
panel['month'] = panel['date'].dt.month
panel['week'] = panel['date'].dt.isocalendar().week.astype(int)
panel['weekday'] = panel['date'].dt.weekday
panel['day_of_year'] = panel['date'].dt.day_of_year

SEASON_MONTHS = list(range(4, 11))
season = panel[panel['month'].isin(SEASON_MONTHS)]

GROUPS = panel['species'].unique().tolist()
COLORS = {
    'болетовые': '#e6194b',
    'лисичковые': '#f58231',
    'лисичковые_трубчатые': '#ffe119',
    'весенние': '#3cb44b',
    'опята': '#4363d8',
}

print(f'Панель: {len(panel)} строк, {panel["date"].nunique()} дней x {panel["species"].nunique()} групп')
print(f'Период: {panel["date"].min().date()} — {panel["date"].max().date()}')
print(f'Группы: {GROUPS}')
print(f'\nСтатистика mushroom_count (сезон):')
print(season.groupby('species')['mushroom_count'].describe().round(1))

## 1. Временные ряды по группам

In [ ]:
# Дневные ряды — каждая группа отдельным графиком
n_groups = len(COLORS)
fig = make_subplots(rows=n_groups, cols=1, shared_xaxes=True, vertical_spacing=0.03,
    subplot_titles=list(COLORS.keys()))

for i, (sp, color) in enumerate(COLORS.items(), 1):
    df_sp = season[season['species'] == sp].sort_values('date')
    fig.add_trace(go.Scatter(
        x=df_sp['date'], y=df_sp['mushroom_count'],
        mode='lines', name=sp,
        line=dict(color=color, width=1),
    ), row=i, col=1)

fig.update_layout(height=200*n_groups, plot_bgcolor='white', showlegend=False,
    title='Дневные ряды по группам')
fig.update_yaxes(gridcolor='#e0e0e0')
fig.update_xaxes(gridcolor='#e0e0e0')
fig.show()

In [ ]:
# Все группы на одном графике — недельная сумма
fig = go.Figure()
for sp, color in COLORS.items():
    df_sp = season[season['species'] == sp].sort_values('date').set_index('date')['mushroom_count']
    weekly = df_sp.resample('W').sum()
    fig.add_trace(go.Scatter(
        x=weekly.index, y=weekly.values,
        mode='lines', name=sp,
        line=dict(color=color, width=2),
    ))

fig.update_layout(
    title='Все группы — недельная сумма',
    xaxis_title='Дата', yaxis_title='mushroom_count / неделю',
    height=500, plot_bgcolor='white', hovermode='x unified',
    yaxis=dict(gridcolor='#e0e0e0'), xaxis=dict(gridcolor='#e0e0e0'),
    legend=dict(orientation='h', y=-0.15),
)
fig.show()

## 2. Сезонность по группам

In [ ]:
# Сезонность по неделям — все 5 групп
n = len(COLORS)
cols = min(n, 3)
rows_n = (n + cols - 1) // cols
fig = make_subplots(rows=rows_n, cols=cols, subplot_titles=list(COLORS.keys()))

for idx, (sp, color) in enumerate(COLORS.items()):
    r, c = divmod(idx, cols)
    df_sp = season[season['species'] == sp]
    by_week = df_sp.groupby('week')['mushroom_count'].mean().round(1)
    by_week = by_week[(by_week.index >= 14) & (by_week.index <= 48)]
    fig.add_trace(go.Bar(
        x=by_week.index, y=by_week.values,
        marker_color=color, name=sp,
    ), row=r+1, col=c+1)

fig.update_layout(height=300*rows_n, plot_bgcolor='white', showlegend=False,
    title='Средняя активность по неделям года')
fig.update_yaxes(gridcolor='#e0e0e0')
fig.update_xaxes(title_text='Неделя')
fig.show()

In [ ]:
# Heatmap: месяц x группа (средний count)
pivot = season.groupby(['month', 'species'])['mushroom_count'].mean().unstack(fill_value=0).round(1)
pivot = pivot[list(COLORS.keys())]  # порядок столбцов

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns,
    y=[f'{m} ({["янв","фев","мар","апр","май","июн","июл","авг","сен","окт","ноя","дек"][m-1]})' for m in pivot.index],
    colorscale='YlOrRd',
    text=pivot.values.round(1),
    texttemplate='%{text}',
))
fig.update_layout(title='Средний count по месяцам и группам', height=400)
fig.show()

## 3. Распределение mushroom_count

In [ ]:
# Распределение ненулевых значений по группам
fig = make_subplots(rows=2, cols=2, subplot_titles=list(COLORS.keys()))
positions = [(1,1), (1,2), (2,1), (2,2)]

for (sp, color), (r, c) in zip(COLORS.items(), positions):
    nonzero = season[(season['species'] == sp) & (season['mushroom_count'] > 0)]['mushroom_count']
    fig.add_trace(go.Histogram(
        x=nonzero, nbinsx=40,
        marker_color=color, name=sp,
    ), row=r, col=c)

fig.update_layout(height=600, plot_bgcolor='white', showlegend=False,
    title='Распределение mushroom_count (только ненулевые дни)')
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()

# Статистика нулей
for sp in COLORS:
    sp_data = season[season['species'] == sp]
    zeros = (sp_data['mushroom_count'] == 0).sum()
    total = len(sp_data)
    nonzero = sp_data[sp_data['mushroom_count'] > 0]['mushroom_count']
    print(f'{sp:15s}: {zeros/total*100:.0f}% нулей, медиана(>0)={nonzero.median():.0f}, p95={nonzero.quantile(0.95):.0f}, max={nonzero.max()}')

## 4. Год к году — сравнение сезонов

In [ ]:
# Каждый год отдельной линией для каждой группы
YEAR_COLORS = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4','#42d4f4','#f032e6','#bfef45']
years = sorted(season['year'].unique())

fig = make_subplots(rows=2, cols=2, subplot_titles=list(COLORS.keys()))
positions = [(1,1), (1,2), (2,1), (2,2)]

for (sp, _), (r, c) in zip(COLORS.items(), positions):
    for j, year in enumerate(years):
        df_yr = season[(season['species'] == sp) & (season['year'] == year)]
        weekly = df_yr.set_index('date')['mushroom_count'].resample('W').sum().reset_index()
        weekly['week'] = weekly['date'].dt.isocalendar().week.astype(int)
        weekly = weekly[(weekly['week'] >= 14) & (weekly['week'] <= 44)]
        fig.add_trace(go.Scatter(
            x=weekly['week'], y=weekly['mushroom_count'],
            mode='lines', name=str(year),
            line=dict(width=2, color=YEAR_COLORS[j % len(YEAR_COLORS)]),
            showlegend=(r == 1 and c == 1),
        ), row=r, col=c)

fig.update_layout(height=700, plot_bgcolor='white',
    title='Сравнение сезонов по годам',
    legend=dict(orientation='h', y=-0.1))
fig.update_yaxes(gridcolor='#e0e0e0')
fig.update_xaxes(title_text='Неделя года', dtick=4)
fig.show()

## 5. Корреляции с погодой

In [ ]:
# Топ-15 погодных фич по корреляции — для каждой группы отдельно
weather_cols = [c for c in panel.columns if c not in ('date', 'species', 'mushroom_count', 'year', 'month', 'week')]

fig = make_subplots(rows=2, cols=2, subplot_titles=list(COLORS.keys()))
positions = [(1,1), (1,2), (2,1), (2,2)]

for (sp, color), (r, c) in zip(COLORS.items(), positions):
    df_sp = season[season['species'] == sp]
    corr = df_sp[weather_cols + ['mushroom_count']].corr()['mushroom_count'].drop('mushroom_count')
    top = corr.abs().nlargest(15)
    top_vals = corr[top.index]
    
    colors_bar = ['#2196F3' if v >= 0 else '#e6194b' for v in top_vals.values]
    fig.add_trace(go.Bar(
        x=top_vals.values, y=top_vals.index,
        orientation='h', marker_color=colors_bar, name=sp,
    ), row=r, col=c)

fig.update_layout(height=900, plot_bgcolor='white', showlegend=False,
    title='Топ-15 корреляций с погодой (для каждой группы отдельно)')
fig.update_xaxes(gridcolor='#e0e0e0')
fig.show()

In [ ]:
# Scatter: топ-фича для каждой группы
fig = make_subplots(rows=2, cols=2, subplot_titles=list(COLORS.keys()))
positions = [(1,1), (1,2), (2,1), (2,2)]

for (sp, color), (r, c) in zip(COLORS.items(), positions):
    df_sp = season[(season['species'] == sp) & (season['mushroom_count'] > 0)]
    corr = df_sp[weather_cols + ['mushroom_count']].corr()['mushroom_count'].drop('mushroom_count')
    top_feat = corr.abs().idxmax()
    
    fig.add_trace(go.Scatter(
        x=df_sp[top_feat], y=df_sp['mushroom_count'],
        mode='markers', marker=dict(color=color, size=4, opacity=0.5),
        name=f'{sp}: {top_feat}',
    ), row=r, col=c)
    fig.update_xaxes(title_text=top_feat, row=r, col=c)

fig.update_layout(height=600, plot_bgcolor='white', showlegend=False,
    title='Scatter: mushroom_count vs топ коррелирующая фича')
fig.update_yaxes(gridcolor='#e0e0e0', title_text='count')
fig.show()

## 6. Соотношение между группами

In [ ]:
# Доля каждой группы по месяцам (stacked area)
monthly = season.groupby(['month', 'species'])['mushroom_count'].sum().unstack(fill_value=0)
monthly = monthly[list(COLORS.keys())]
monthly_pct = monthly.div(monthly.sum(axis=1), axis=0) * 100

fig = go.Figure()
for sp, color in COLORS.items():
    fig.add_trace(go.Scatter(
        x=monthly_pct.index, y=monthly_pct[sp],
        mode='lines', name=sp,
        line=dict(width=0), fill='tonexty', fillcolor=color.replace(')', ',0.5)').replace('rgb', 'rgba'),
        stackgroup='one',
    ))

month_names = ['','','','Апр','Май','Июн','Июл','Авг','Сен','Окт']
fig.update_layout(
    title='Доля каждой группы по месяцам (%)',
    xaxis=dict(title='Месяц', tickvals=list(range(4,11)), ticktext=month_names[4:]),
    yaxis=dict(title='%', gridcolor='#e0e0e0'),
    height=450, plot_bgcolor='white',
    legend=dict(orientation='h', y=-0.15),
)
fig.show()

In [ ]:
# Суммарный сбор по годам и группам (stacked bar)
yearly = season.groupby(['year', 'species'])['mushroom_count'].sum().unstack(fill_value=0)
yearly = yearly[list(COLORS.keys())]

fig = go.Figure()
for sp, color in COLORS.items():
    fig.add_trace(go.Bar(
        x=yearly.index, y=yearly[sp],
        name=sp, marker_color=color,
    ))

fig.update_layout(
    title='Суммарный сбор по годам и группам',
    barmode='stack',
    xaxis_title='Год', yaxis_title='Суммарный mushroom_count',
    height=450, plot_bgcolor='white',
    yaxis=dict(gridcolor='#e0e0e0'),
    legend=dict(orientation='h', y=-0.15),
)
fig.show()

## 7. Связь между группами — растут ли вместе?

In [ ]:
# Корреляция между группами (по дневным данным)
pivot_daily = season.pivot_table(index='date', columns='species', values='mushroom_count', fill_value=0)
corr_matrix = pivot_daily.corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.index,
    colorscale='RdBu_r', zmin=-1, zmax=1,
    text=corr_matrix.values,
    texttemplate='%{text:.2f}',
))
fig.update_layout(title='Корреляция между группами (по дням)', height=400, width=500)
fig.show()

print('Корреляция болетовые vs лисичковые:', corr_matrix.loc['болетовые', 'лисичковые'])
print('Корреляция болетовые vs опята:', corr_matrix.loc['болетовые', 'опята'])

## 8. Пиковые дни

In [ ]:
# Топ-10 дней для каждой группы
for sp in COLORS:
    df_sp = season[season['species'] == sp]
    top = df_sp.nlargest(10, 'mushroom_count')[['date', 'mushroom_count']]
    dates_str = ', '.join(f"{r['date'].strftime('%d.%m.%Y')}={int(r['mushroom_count'])}" for _, r in top.iterrows())
    print(f'{sp}: {dates_str}')
    print()

## 9. Погода в дни пиков vs обычные дни

In [ ]:
# Сравнение погоды: пиковые дни (top 10%) vs обычные — для каждой группы
key_weather = ['temperature_2m_mean', 'precipitation_sum', 'relative_humidity_2m',
               'soil_temperature_0cm', 'soil_moisture_0_to_1cm', 'sunshine_hours']
key_weather = [c for c in key_weather if c in panel.columns]

for sp in COLORS:
    df_sp = season[season['species'] == sp]
    nonzero = df_sp[df_sp['mushroom_count'] > 0]
    if len(nonzero) < 20:
        continue
    threshold = nonzero['mushroom_count'].quantile(0.9)
    peak = nonzero[nonzero['mushroom_count'] >= threshold]
    normal = nonzero[nonzero['mushroom_count'] < threshold]

    print(f'\n=== {sp} (пик >= {threshold:.0f}, дней: {len(peak)}) ===')
    for col in key_weather:
        p_mean = peak[col].mean()
        n_mean = normal[col].mean()
        diff = ((p_mean - n_mean) / n_mean * 100) if n_mean != 0 else 0
        print(f'  {col:30s}: пик={p_mean:.1f}, обычн={n_mean:.1f}, разница={diff:+.0f}%')

## 10. Температура vs mushroom_count — по группам и месяцам

In [ ]:
# Scatter: температура vs count, раскрашенный по месяцам — для каждой группы
if 'temperature_2m_mean' in panel.columns:
    fig = make_subplots(rows=2, cols=3, subplot_titles=list(COLORS.keys()) + [''])

    positions = [(1,1), (1,2), (1,3), (2,1), (2,2)]
    for (sp, color), (r, c) in zip(COLORS.items(), positions):
        df_sp = season[(season['species'] == sp) & (season['mushroom_count'] > 0)]
        fig.add_trace(go.Scatter(
            x=df_sp['temperature_2m_mean'], y=df_sp['mushroom_count'],
            mode='markers', name=sp,
            marker=dict(color=df_sp['month'], colorscale='Viridis', size=4, opacity=0.5,
                        colorbar=dict(title='Месяц') if r==1 and c==3 else None),
        ), row=r, col=c)
        fig.update_xaxes(title_text='Temp mean', row=r, col=c)

    fig.update_layout(height=600, plot_bgcolor='white', showlegend=False,
        title='Температура vs mushroom_count (цвет = месяц)')
    fig.update_yaxes(gridcolor='#e0e0e0')
    fig.show()

## 11. Осадки и влажность почвы — лаговый эффект

In [ ]:
# Корреляция mushroom_count с осадками за разные лаги — какой лаг лучше?
precip_cols = [c for c in panel.columns if c.startswith('precip_') and 'd' in c and not c.startswith('precip_lag')]
precip_cols = sorted(precip_cols, key=lambda x: int(''.join(filter(str.isdigit, x))) if any(c.isdigit() for c in x) else 0)

fig = make_subplots(rows=2, cols=3, subplot_titles=list(COLORS.keys()) + [''])
positions = [(1,1), (1,2), (1,3), (2,1), (2,2)]

for (sp, color), (r, c) in zip(COLORS.items(), positions):
    df_sp = season[season['species'] == sp]
    corrs = []
    for col in precip_cols:
        if col in df_sp.columns:
            corrs.append({'col': col, 'corr': df_sp[[col, 'mushroom_count']].corr().iloc[0,1]})
    if corrs:
        corr_df = pd.DataFrame(corrs)
        fig.add_trace(go.Bar(
            x=corr_df['col'], y=corr_df['corr'],
            marker_color=color, name=sp,
        ), row=r, col=c)

fig.update_layout(height=600, plot_bgcolor='white', showlegend=False,
    title='Корреляция с накопленными осадками (какое окно лучше?)')
fig.update_yaxes(gridcolor='#e0e0e0')
fig.update_xaxes(tickangle=45)
fig.show()

## 12. GDD (градусо-дни) vs грибы

In [ ]:
# GDD (градусо-дни) vs mushroom_count — scatter для каждой группы
gdd_cols = [c for c in panel.columns if c.startswith('gdd_')]
if gdd_cols:
    best_gdd = 'gdd_14d' if 'gdd_14d' in gdd_cols else gdd_cols[0]

    fig = make_subplots(rows=2, cols=3, subplot_titles=list(COLORS.keys()) + [''])
    positions = [(1,1), (1,2), (1,3), (2,1), (2,2)]

    for (sp, color), (r, c) in zip(COLORS.items(), positions):
        df_sp = season[(season['species'] == sp) & (season['mushroom_count'] > 0)]
        fig.add_trace(go.Scatter(
            x=df_sp[best_gdd], y=df_sp['mushroom_count'],
            mode='markers', marker=dict(color=color, size=4, opacity=0.4),
            name=sp,
        ), row=r, col=c)
        fig.update_xaxes(title_text=best_gdd, row=r, col=c)

    fig.update_layout(height=600, plot_bgcolor='white', showlegend=False,
        title=f'{best_gdd} (градусо-дни) vs mushroom_count')
    fig.update_yaxes(gridcolor='#e0e0e0')
    fig.show()

## 13. Средний сезонный профиль по дню года

In [ ]:
# Средний профиль по дню года — все группы на одном графике
fig = go.Figure()
for sp, color in COLORS.items():
    df_sp = season[season['species'] == sp]
    by_doy = df_sp.groupby('day_of_year')['mushroom_count'].mean()
    # Сглаживание 7 дней для читаемости
    smoothed = by_doy.rolling(7, center=True, min_periods=1).mean()
    fig.add_trace(go.Scatter(
        x=smoothed.index, y=smoothed.values,
        mode='lines', name=sp,
        line=dict(color=color, width=2.5),
    ))

fig.update_layout(
    title='Средний сезонный профиль (mushroom_count по дню года, сглажен 7 дней)',
    xaxis_title='День года', yaxis_title='Среднее mushroom_count',
    height=450, plot_bgcolor='white', hovermode='x unified',
    yaxis=dict(gridcolor='#e0e0e0'), xaxis=dict(gridcolor='#e0e0e0', dtick=30),
    legend=dict(orientation='h', y=-0.15),
)
fig.show()

## 14. Boxplot по месяцам для каждой группы

In [ ]:
# Boxplot по месяцам — только ненулевые дни
fig = make_subplots(rows=2, cols=3, subplot_titles=list(COLORS.keys()) + [''])
positions = [(1,1), (1,2), (1,3), (2,1), (2,2)]

for (sp, color), (r, c) in zip(COLORS.items(), positions):
    df_sp = season[(season['species'] == sp) & (season['mushroom_count'] > 0)]
    fig.add_trace(go.Box(
        x=df_sp['month'], y=df_sp['mushroom_count'],
        marker_color=color, name=sp,
    ), row=r, col=c)
    fig.update_xaxes(title_text='Месяц', dtick=1, row=r, col=c)

fig.update_layout(height=600, plot_bgcolor='white', showlegend=False,
    title='Распределение mushroom_count по месяцам (ненулевые дни)')
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()

## 15. Паттерн "дождь → тепло → грибы"

In [ ]:
# Средний count в зависимости от "дождь N дней назад + тепло сейчас"
rain_warm_cols = [c for c in panel.columns if 'rain' in c and 'then_warm' in c and 'lag' not in c]

if rain_warm_cols:
    fig = make_subplots(rows=2, cols=3, subplot_titles=list(COLORS.keys()) + [''])
    positions = [(1,1), (1,2), (1,3), (2,1), (2,2)]

    for (sp, color), (r, c) in zip(COLORS.items(), positions):
        df_sp = season[season['species'] == sp]
        means = []
        for col in rain_warm_cols:
            if col in df_sp.columns:
                m_yes = df_sp.loc[df_sp[col] == 1, 'mushroom_count'].mean()
                m_no = df_sp.loc[df_sp[col] == 0, 'mushroom_count'].mean()
                means.append({'pattern': col.replace('rain','R').replace('d_then_warm','->W'),
                             'with': m_yes, 'without': m_no})
        if means:
            mdf = pd.DataFrame(means)
            fig.add_trace(go.Bar(x=mdf['pattern'], y=mdf['with'], marker_color=color, name='с паттерном'), row=r, col=c)
            fig.add_trace(go.Bar(x=mdf['pattern'], y=mdf['without'], marker_color='#ccc', name='без'), row=r, col=c)

    fig.update_layout(height=600, plot_bgcolor='white', showlegend=False, barmode='group',
        title='Средний count: с паттерном "дождь→тепло" (цвет) vs без (серый)')
    fig.update_yaxes(gridcolor='#e0e0e0')
    fig.show()
else:
    print('Фичи rain_then_warm не найдены')

## 16. Стабильность по годам — CV коэффициент

In [ ]:
# Насколько стабилен сезон от года к году?
yearly_totals = season.groupby(['year', 'species'])['mushroom_count'].sum().unstack(fill_value=0)

fig = go.Figure()
for sp, color in COLORS.items():
    if sp in yearly_totals.columns:
        vals = yearly_totals[sp]
        cv = vals.std() / vals.mean() * 100 if vals.mean() > 0 else 0
        fig.add_trace(go.Bar(
            x=yearly_totals.index.astype(str), y=vals,
            name=f'{sp} (CV={cv:.0f}%)',
            marker_color=color,
        ))

fig.update_layout(
    title='Суммарный сезонный сбор по годам (с коэффициентом вариации)',
    barmode='group', height=450, plot_bgcolor='white',
    yaxis=dict(gridcolor='#e0e0e0'), xaxis_title='Год',
    legend=dict(orientation='h', y=-0.15),
)
fig.show()

# Таблица
print('\nСуммарный сбор по годам:')
print(yearly_totals.round(0).to_string())

## 17. Отбор фич: Спирмен корреляция по группам

Для каждой группы — только в её сезоне. Оставляем 25 фич с наибольшей абсолютной корреляцией Спирмена.

In [ ]:
from scipy.stats import spearmanr
import json, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

GROUP_SEASONS = {
    'болетовые':              list(range(6, 11)),
    'лисичковые':             list(range(6, 10)),
    'лисичковые_трубчатые':   [10, 11],
    'весенние':               [4, 5],
    'опята':                  list(range(6, 11)),
}

# Исключаем из кандидатов
EXCLUDE = {'date', 'species', 'mushroom_count', 'year', 'month', 'week', 'weekday',
           'report_count', 'mushroom_index', 'mushroom_index_sm', 'audience_scale',
           'avg_likes', 'total_photos', 'avg_views', 'median_views', 'audience_proxy',
           'vk_trend', 'weekday_scale', 'is_weekend', 'is_holiday', 'is_day_off', 'long_weekend',
           'day_of_month', 'is_around_20th_may', 'is_around_20th_jun', 'is_around_20th_jul',
           'is_around_20th_aug', 'is_around_20th_sep', 'is_peak_window',
           'days_to_aug20', 'days_to_sep20', 'days_to_peak', 'season_gauss'}

# Фурье — оставляем только 1-ю гармонику
EXCLUDE_FOURIER = {f'sin_doy{k}' for k in range(2, 6)} | {f'cos_doy{k}' for k in range(2, 6)}
EXCLUDE = EXCLUDE | EXCLUDE_FOURIER

N_TOP = 25

candidate_cols = [c for c in panel.columns if c not in EXCLUDE and panel[c].dtype in ('float64', 'int64', 'float32', 'int32')]
print(f'Кандидатов на фичи: {len(candidate_cols)}')

# Всегда включаем
ALWAYS_INCLUDE = ['day_of_year', 'month', 'sin_doy1', 'cos_doy1']

selected_features = {}

for g, months in GROUP_SEASONS.items():
    df_g = panel[(panel['species'] == g) & (panel['date'].dt.month.isin(months))].copy()
    
    # Спирмен для каждой фичи — пропускаем константные
    correlations = {}
    skipped_const = 0
    for col in candidate_cols:
        if col not in df_g.columns:
            continue
        vals = df_g[[col, 'mushroom_count']].dropna()
        if len(vals) < 30:
            continue
        if vals[col].std() == 0:
            skipped_const += 1
            continue
        rho, pval = spearmanr(vals[col], vals['mushroom_count'])
        if not np.isnan(rho):
            correlations[col] = {'rho': rho, 'abs_rho': abs(rho), 'pval': pval}
    
    corr_df = pd.DataFrame(correlations).T.sort_values('abs_rho', ascending=False)
    
    # Топ-N + всегда включённые
    top_feats = corr_df.head(N_TOP).index.tolist()
    final_feats = list(dict.fromkeys(ALWAYS_INCLUDE + top_feats))
    selected_features[g] = final_feats
    
    print(f'\n=== {g} (месяцы {months}, строк: {len(df_g)}, const={skipped_const}) ===')
    print(f'Отобрано: {len(final_feats)} фич')
    for i, feat in enumerate(final_feats):
        rho = correlations.get(feat, {}).get('rho', 0)
        marker = ' *' if feat in ALWAYS_INCLUDE else ''
        print(f'  {i+1:2d}. {feat:35s} rho={rho:+.3f}{marker}')

# Сохраняем
with open('data/selected_features.json', 'w', encoding='utf-8') as f:
    json.dump(selected_features, f, ensure_ascii=False, indent=2)
print(f'\nСохранено: data/selected_features.json')

In [ ]:
# Визуализация отобранных фич по группам
cols = min(len(GROUP_SEASONS), 3)
rows_n = (len(GROUP_SEASONS) + cols - 1) // cols
fig = make_subplots(rows=rows_n, cols=cols, subplot_titles=list(GROUP_SEASONS.keys()))

for idx, (g, months) in enumerate(GROUP_SEASONS.items()):
    r, c = divmod(idx, cols)
    df_g = panel[(panel['species'] == g) & (panel['date'].dt.month.isin(months))]
    
    feats = selected_features[g]
    rhos = []
    for feat in feats:
        if feat in df_g.columns:
            vals = df_g[[feat, 'mushroom_count']].dropna()
            if len(vals) > 10:
                rho, _ = spearmanr(vals[feat], vals['mushroom_count'])
                rhos.append(rho)
            else:
                rhos.append(0)
        else:
            rhos.append(0)
    
    colors = ['#2196F3' if v >= 0 else '#e6194b' for v in rhos]
    fig.add_trace(go.Bar(
        x=rhos, y=feats,
        orientation='h', marker_color=colors,
        showlegend=False,
    ), row=r+1, col=c+1)

fig.update_layout(height=500*rows_n, plot_bgcolor='white',
    title=f'Отобранные фичи: корреляция Спирмена с mushroom_count (по {N_TOP} на группу)')
fig.update_xaxes(gridcolor='#e0e0e0', title_text='rho')
fig.show()

# Общие фичи между группами
from collections import Counter
all_feats = Counter()
for feats in selected_features.values():
    all_feats.update(feats)

print('\nФичи, отобранные для нескольких групп:')
for f, cnt in all_feats.most_common():
    if cnt >= 2:
        groups_with = [g for g, feats in selected_features.items() if f in feats]
        print(f'  {f:35s} [{cnt} групп]: {", ".join(groups_with)}')